<a href="https://colab.research.google.com/github/mariangelesalomar-sudo/eigenfaces-dma-grupo-1/blob/main/eigenfaces_caras_clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reconocimiento facial con Eigenfaces (PCA)

**Maestría en Ciencia de Datos — Data Mining Avanzado — 2026 / 2° Cuatrimestre**

Trabajo grupal: detección y comparación de las caras de 14 alumnos de la clase usando
el método clásico de **Eigenfaces** (Análisis de Componentes Principales).

## Pipeline
1. Carga de fotos desde Google Drive (formatos `.jpg`, `.jpeg`, `.png`, `.heic`)
2. Detección de rostros con **DeepFace (RetinaFace)**, con respaldo en Haar Cascade
3. Recorte por borde oval del rostro y conversión a escala de grises de 30×30 px
4. Reducción de dimensionalidad con PCA → Eigenfaces
5. Cálculo del vector promedio por alumno
6. Matriz de distancias euclidianas entre alumnos
7. Ranking de pares más parecidos / menos parecidos

## Notas importantes
- Las fotos **no** se incluyen en este repositorio por privacidad. Cada integrante del grupo
  debe tener acceso a la carpeta compartida de Google Drive.
- El notebook está pensado para correr en **Google Colab**.


## 1. Instalar librerías externas

In [1]:
# opencv-python-headless → detectar y recortar caras
# scikit-learn           → hacer PCA (Eigenfaces)
# pillow-heif            → leer fotos .HEIC de iPhone
# matplotlib             → hacer gráficos
# scipy                  → calcular distancias entre vectores

!pip install opencv-python-headless scikit-learn pillow-heif matplotlib scipy Pillow --quiet

print("✅ Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 32.1 MB/s eta 0:00:00
✅ Librerías instaladas correctamente


## 2. Importar librerías al entorno de trabajo

In [12]:
import os                          # para leer archivos del disco
import cv2                         # para procesar imágenes y detectar caras
import numpy as np                 # para trabajar con vectores y matrices
import matplotlib.pyplot as plt    # para hacer gráficos y mostrar imágenes
from PIL import Image              # para abrir imágenes (incluyendo HEIC)
from sklearn.decomposition import PCA      # para hacer Eigenfaces
from scipy.spatial.distance import euclidean  # para calcular distancias
from google.colab import drive     # para conectarse a Google Drive

# Activar soporte para fotos .HEIC (formato iPhone)
from pillow_heif import register_heif_opener
register_heif_opener()

import warnings
warnings.filterwarnings('ignore')  # silenciar advertencias menores

print("✅ Todo importado. Estamos listos para trabajar.")

✅ Todo importado. Estamos listos para trabajar.


## 3. Conectar a Google Drive

In [13]:
# Le decimos a Colab que queremos acceder a nuestro Drive.
# Va a pedir permiso la primera vez (aparece un botón).

drive.mount('/content/drive')

# Definimos la ruta exacta donde están las fotos
ruta_fotos = '/content/drive/MyDrive/caras'

# Verificamos que la ruta existe y contamos los archivos
archivos_totales = os.listdir(ruta_fotos)

print(f"Drive conectado correctamente")
print(f"Carpeta: {ruta_fotos}")
print(f"Cantidad total de archivos en la carpeta: {len(archivos_totales)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive conectado correctamente
Carpeta: /content/drive/MyDrive/caras
Cantidad total de archivos en la carpeta: 2153


## 4. Definir a qué alumno pertenece cada foto

In [ ]:
# PROBLEMA: todas las fotos están en una sola carpeta,
# y el nombre del alumno está dentro del nombre del archivo.
# Además, el mismo alumno aparece escrito de formas distintas
# (typos, mayúsculas, guiones extra, etc.)

MAPA_NOMBRES = {
    # Juani (4 variantes)
    'juani_cahionne'    : 'juani_cacchione',
    'juani_cachionne'   : 'juani_cacchione',
    'juani_cachione'    : 'juani_cacchione',
    'juani_cacchione'   : 'juani_cacchione',
    'juan_cacchione'    : 'juani_cacchione',
    # Mariángeles (2 variantes)
    'mariangeles_alomar': 'mariangeles_alomar',
    'mariangeles_aloar' : 'mariangeles_alomar',
    # Miguel (2 variantes)
    'miguel_garrone'    : 'miguel_garrone',
    'migue_garrone'     : 'miguel_garrone',
    # El resto (sin variantes)
    'agustina_sebben'   : 'agustina_sebben',
    'belen_maldonado'   : 'belen_maldonado',
    'fede_spinelli'     : 'fede_spinelli',
    'guillermo_anso'    : 'guillermo_anso',
    'judi_luna'         : 'judi_luna',
    'judi_luna_'        : 'judi_luna',
    'lucia_tamplin'     : 'lucia_tamplin',
    'Lucia_Tamplin'     : 'lucia_tamplin',
    'martin_ceriotti'   : 'martin_ceriotti',
    'matias_villanueva' : 'matias_villanueva',
    'millie_teran'      : 'millie_teran',
    'tomas_delbo'       : 'tomas_delbo',
}

def extraer_nombre_alumno(nombre_archivo):
    """Dado el nombre de un archivo, devuelve el nombre canónico del alumno."""
    sin_extension = os.path.splitext(nombre_archivo)[0]
    # Recorremos los prefijos del más largo al más corto
    prefijos = sorted(MAPA_NOMBRES.keys(), key=len, reverse=True)
    for prefijo in prefijos:
        if sin_extension.startswith(prefijo):
            return MAPA_NOMBRES[prefijo]
    return None  # archivo no reconocido

# ── Prueba visual ──────────────────────────────────────────
print("Probando la función con algunos ejemplos:\n")
ejemplos = [
    'martin_ceriotti (5).jpg',
    'belen_maldonado_001.heic',
    'juani_cahionne (3).jpeg',
    'juani_cacchione (1).heic',
    'migue_garrone (1).MOV',
    'Lucia_Tamplin_03.jpeg',
    'judi_luna_ (2).jpg',
]
for e in ejemplos:
    resultado = extraer_nombre_alumno(e)
    print(f"   '{e}'  →  '{resultado}'")

print("\nLa función reconoce correctamente los nombres")

## 5. Abrir y mostrar UNA foto original

In [ ]:
# Antes de procesar todo, vamos a ver cómo se ve una foto
# tal cual sale de la cámara, sin ninguna modificación.

alumno_ejemplo = 'martin_ceriotti'

foto_ejemplo = None
for archivo in sorted(os.listdir(ruta_fotos)):
    if extraer_nombre_alumno(archivo) == alumno_ejemplo:
        ext = os.path.splitext(archivo)[1].lower()
        if ext in ('.jpg', '.jpeg', '.png', '.heic'):
            foto_ejemplo = archivo
            break

ruta_ejemplo = os.path.join(ruta_fotos, foto_ejemplo)

# Leer la imagen (soporte para HEIC y formatos comunes)
def leer_imagen(ruta):
    ext = os.path.splitext(ruta)[1].lower()
    if ext == '.heic':
        img_pil = Image.open(ruta).convert('RGB')
        return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    else:
        return cv2.imread(ruta)

imagen_original = leer_imagen(ruta_ejemplo)
imagen_rgb = cv2.cvtColor(imagen_original, cv2.COLOR_BGR2RGB)

print(f"Foto original de {alumno_ejemplo}: '{foto_ejemplo}'")
print(f"   Dimensiones: {imagen_original.shape[1]} px de ancho × {imagen_original.shape[0]} px de alto")
print(f"   Canales de color: {imagen_original.shape[2]} (R, G, B)")

plt.figure(figsize=(4, 5))
plt.imshow(imagen_rgb)
plt.title(f"PASO 0: Foto original\n{alumno_ejemplo}", fontsize=11, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Recorte por borde real del rostro con DeepFace

In [ ]:
# DeepFace detecta la cara Y sus landmarks (puntos clave).
# Con esos puntos podemos dibujar una MÁSCARA ELÍPTICA que sigue el óvalo
# de la cara y recortar SOLO lo que está dentro del óvalo (fondo negro afuera).

!pip install deepface --quiet
print("✅ DeepFace instalado")

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image
from deepface import DeepFace

print("✅ DeepFace importado\n")


def recortar_cara_por_borde(imagen_bgr, margen=0.18):
    """
    Detecta la cara con DeepFace y recorta siguiendo el borde
    real del rostro (forma elíptica), no un rectángulo rígido.
    """
    alto, ancho = imagen_bgr.shape[:2]
    imagen_rgb  = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

    try:
        caras = DeepFace.extract_faces(
            img_path          = imagen_bgr,
            detector_backend  = 'retinaface',
            enforce_detection = False,
            align             = True
        )

        caras_validas = [c for c in caras if c['confidence'] > 0]
        if not caras_validas:
            return None, None, None

        mejor     = max(caras_validas, key=lambda c: c['confidence'])
        confianza = mejor['confidence']

        region = mejor['facial_area']
        x, y   = region['x'], region['y']
        w, h   = region['w'], region['h']

        landmarks = region
        ojo_izq   = landmarks.get('left_eye')
        ojo_der   = landmarks.get('right_eye')
        nariz     = landmarks.get('nose')
        boca_izq  = landmarks.get('mouth_left')
        boca_der  = landmarks.get('mouth_right')

        # Centro: punto medio entre los dos ojos
        if ojo_izq and ojo_der:
            cx = int((ojo_izq[0] + ojo_der[0]) / 2)
            cy = int((ojo_izq[1] + ojo_der[1]) / 2)
        else:
            cx = x + w // 2
            cy = y + h // 2

        # Eje horizontal: distancia entre ojos × factor de margen
        if ojo_izq and ojo_der:
            dist_ojos  = np.sqrt((ojo_der[0]-ojo_izq[0])**2 +
                                 (ojo_der[1]-ojo_izq[1])**2)
            radio_x    = int(dist_ojos * 1.5 * (1 + margen))
        else:
            radio_x    = int(w * 0.55 * (1 + margen))

        # Eje vertical
        if ojo_izq and ojo_der and boca_izq and boca_der:
            cy_boca    = int((boca_izq[1] + boca_der[1]) / 2)
            dist_ojo_boca = abs(cy_boca - cy)
            radio_y_abajo  = int(dist_ojo_boca * 1.3 * (1 + margen))
            radio_y_arriba = int(dist_ojo_boca * 1.6 * (1 + margen))
        else:
            radio_y_abajo  = int(h * 0.55 * (1 + margen))
            radio_y_arriba = int(h * 0.55 * (1 + margen))

        # Ángulo de inclinación de la cara
        if ojo_izq and ojo_der:
            angulo = np.degrees(np.arctan2(
                ojo_der[1] - ojo_izq[1],
                ojo_der[0] - ojo_izq[0]
            ))
        else:
            angulo = 0

        # Recorte rectangular amplio (zona de trabajo)
        margen_rect = int(max(radio_x, radio_y_abajo, radio_y_arriba) * 1.1)
        rx1 = max(0,     cx - margen_rect)
        ry1 = max(0,     cy - margen_rect)
        rx2 = min(ancho, cx + margen_rect)
        ry2 = min(alto,  cy + margen_rect)

        recorte_rgb = imagen_rgb[ry1:ry2, rx1:rx2].copy()
        rh, rw      = recorte_rgb.shape[:2]

        cx_local = cx - rx1
        cy_local = cy - ry1

        radio_y_prom = (radio_y_arriba + radio_y_abajo) // 2

        mascara = np.zeros((rh, rw), dtype=np.uint8)
        cv2.ellipse(
            mascara,
            center    = (cx_local, cy_local),
            axes      = (radio_x, radio_y_prom),
            angle     = angulo,
            startAngle= 0,
            endAngle  = 360,
            color     = 255,
            thickness = -1
        )

        mascara_suave = cv2.GaussianBlur(mascara, (21, 21), 0)
        mascara_3ch = np.stack([mascara_suave]*3, axis=-1).astype(np.float32) / 255.0
        cara_con_mascara = (recorte_rgb.astype(np.float32) * mascara_3ch).astype(np.uint8)

        ys, xs = np.where(mascara > 0)
        if len(xs) == 0 or len(ys) == 0:
            return None, None, None

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        cara_recortada = cara_con_mascara[y_min:y_max, x_min:x_max]

        cara_gris       = cv2.cvtColor(cara_recortada, cv2.COLOR_RGB2GRAY)
        cara_gris_30x30 = cv2.resize(cara_gris, (30, 30))

        return cara_gris_30x30, cara_recortada, confianza

    except Exception as e:
        return None, None, None


# Haar como respaldo
clasificador_haar = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

## 7. Procesar TODAS las fotos (con checkpoint)

In [ ]:
# MEJORAS clave:
#   1. BARRA DE PROGRESO
#   2. DEDUPLICACIÓN: si hay misma foto en .heic Y .jpeg, usa solo la .jpeg
#   3. GUARDADO PARCIAL cada 50 fotos (checkpoint en Drive)
#   4. RETOMAR DESDE CHECKPOINT si Colab se reinicia
#   5. DeepFace + Haar como respaldo

import numpy as np
import os
import cv2
import time
from PIL import Image

EXTENSIONES     = ('.jpg', '.jpeg', '.png', '.heic')
GUARDAR_CADA    = 50
ruta_checkpoint = '/content/.drive/MyDrive/DMA/datos_procesados/checkpoint.npz'

def leer_imagen(ruta):
    ext = os.path.splitext(ruta)[1].lower()
    if ext == '.heic':
        try:
            img_pil = Image.open(ruta).convert('RGB')
            return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        except:
            return None
    else:
        return cv2.imread(ruta)

clasificador_haar = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# ── Construir lista de archivos a procesar ──────────────────
print("📋 Analizando archivos disponibles...\n")

archivos_todos = sorted(os.listdir(ruta_fotos))
candidatos     = {}

for archivo in archivos_todos:
    ext = os.path.splitext(archivo)[1].lower()
    if ext not in EXTENSIONES:
        continue
    alumno = extraer_nombre_alumno(archivo)
    if alumno is None:
        continue
    base  = os.path.splitext(archivo)[0]
    clave = (alumno, base)
    if clave not in candidatos:
        candidatos[clave] = archivo
    else:
        ext_existente = os.path.splitext(candidatos[clave])[1].lower()
        if ext_existente == '.heic' and ext != '.heic':
            candidatos[clave] = archivo

archivos_a_procesar = sorted(candidatos.values())

# ── Retomar desde checkpoint si existe ──────────────────────
todos_vectores         = []
todas_etiquetas        = []
todas_imagenes         = []
imagen_repr            = {}
ruta_repr              = {}
conteo                 = {}
archivos_ya_procesados = set()

os.makedirs(os.path.dirname(ruta_checkpoint), exist_ok=True)

if os.path.exists(ruta_checkpoint):
    print(f"♻️  Checkpoint encontrado. Retomando...")
    cp = np.load(ruta_checkpoint, allow_pickle=True)
    todos_vectores         = list(cp['vectores'])
    todas_etiquetas        = list(cp['etiquetas'])
    archivos_ya_procesados = set(cp['archivos_procesados'])

    for v, e in zip(todos_vectores, todas_etiquetas):
        conteo[e] = conteo.get(e, 0) + 1
        if e not in imagen_repr:
            imagen_repr[e] = v.reshape(30, 30).astype(np.uint8)

    todas_imagenes = [v.reshape(30, 30).astype(np.uint8) for v in todos_vectores]

# ── Procesar ────────────────────────────────────────────────
archivos_pendientes = [a for a in archivos_a_procesar
                       if a not in archivos_ya_procesados]

total       = len(archivos_pendientes)
sin_cara    = 0
errores     = 0
uso_haar    = 0
inicio      = time.time()

print(f"🔍 Procesando {total} archivos pendientes...\n")

for i, archivo in enumerate(archivos_pendientes):
    if i % 10 == 0 or i == total - 1:
        pct      = (i + 1) / total
        barras   = int(pct * 30)
        barra    = "█" * barras + "░" * (30 - barras)
        elapsed  = time.time() - inicio
        restante = (elapsed / (i + 1)) * (total - i - 1) if i > 0 else 0
        print(f"\r   [{barra}] {i+1}/{total}  "
              f"({pct*100:.0f}%)  ~{restante/60:.1f} min restantes",
              end='', flush=True)

    alumno = extraer_nombre_alumno(archivo)
    ruta   = os.path.join(ruta_fotos, archivo)
    imagen = leer_imagen(ruta)
    if imagen is None:
        errores += 1
        archivos_ya_procesados.add(archivo)
        continue

    cara_30x30, _, _ = recortar_cara_por_borde(imagen)

    if cara_30x30 is None:
        gris  = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        caras = clasificador_haar.detectMultiScale(
            gris, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40)
        )
        if len(caras) > 0:
            x, y, w, h = sorted(caras, key=lambda c: c[2]*c[3], reverse=True)[0]
            cara_30x30 = cv2.resize(gris[y:y+h, x:x+w], (30, 30))
            uso_haar  += 1
        else:
            sin_cara += 1
            archivos_ya_procesados.add(archivo)
            continue

    vector = cara_30x30.flatten().astype(float)
    todos_vectores.append(vector)
    todas_etiquetas.append(alumno)
    todas_imagenes.append(cara_30x30)
    conteo[alumno] = conteo.get(alumno, 0) + 1
    archivos_ya_procesados.add(archivo)

    if alumno not in imagen_repr:
        imagen_repr[alumno] = cara_30x30
        ruta_repr[alumno]   = ruta

    if len(todos_vectores) % GUARDAR_CADA == 0:
        np.savez_compressed(
            ruta_checkpoint,
            vectores            = np.array(todos_vectores),
            etiquetas           = np.array(todas_etiquetas),
            archivos_procesados = np.array(list(archivos_ya_procesados))
        )

np.savez_compressed(
    ruta_checkpoint,
    vectores            = np.array(todos_vectores),
    etiquetas           = np.array(todas_etiquetas),
    archivos_procesados = np.array(list(archivos_ya_procesados))
)

matriz_vectores = np.array(todos_vectores)
etiquetas       = np.array(todas_etiquetas)
nombres_alumnos = sorted(conteo.keys())

print(f"\n\n📊 RESUMEN: {len(todos_vectores)} fotos procesadas")
print(f"   Sin cara: {sin_cara}  |  Errores: {errores}  |  Haar respaldo: {uso_haar}")

## 8. PCA — Eigenfaces

In [ ]:
# Reducimos las 900 dimensiones a ~50 'eigenfaces' (componentes principales).

n_componentes = min(len(todos_vectores) - 1, 50)

print(f"Aplicando PCA: {len(todos_vectores)} fotos × 900 dimensiones → {n_componentes} eigenfaces\n")

pca = PCA(n_components=n_componentes, whiten=True)
pca.fit(matriz_vectores)

fotos_proyectadas = pca.transform(matriz_vectores)

varianza = np.cumsum(pca.explained_variance_ratio_) * 100
print(f"✅ Con {n_componentes} eigenfaces capturamos el {varianza[-1]:.1f}% de la información")

## 9. Vector promedio por alumno y matriz de distancias

In [ ]:
# Cada alumno tiene varias fotos. Calculamos el PROMEDIO de sus vectores
# para representarlo con UN solo punto en el espacio de eigenfaces.

vectores_promedio = {}
for nombre in nombres_alumnos:
    indices = [i for i, e in enumerate(etiquetas) if e == nombre]
    proyecciones = fotos_proyectadas[indices]
    vectores_promedio[nombre] = proyecciones.mean(axis=0)

# Matriz de distancias euclidianas
n = len(nombres_alumnos)
matriz_dist = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            matriz_dist[i][j] = euclidean(
                vectores_promedio[nombres_alumnos[i]],
                vectores_promedio[nombres_alumnos[j]]
            )

print("✅ Matriz de distancias calculada")

## 10. Resultado final — Ranking de similitud

In [ ]:
print("=" * 65)
print("🏆 RANKING DE SIMILITUD")
print("=" * 65)

pares = []
for i in range(n):
    for j in range(i + 1, n):
        pares.append({
            'a1': nombres_alumnos[i],
            'a2': nombres_alumnos[j],
            'd' : matriz_dist[i][j]
        })

pares_ord = sorted(pares, key=lambda x: x['d'])

print("\n🥇 Los 5 pares MÁS PARECIDOS:")
for k, p in enumerate(pares_ord[:5], 1):
    print(f"   {k}. {p['a1']}  ↔  {p['a2']}  (dist: {p['d']:.2f})")

print("\n💨 Los 5 pares MENOS PARECIDOS:")
for k, p in enumerate(pares_ord[-5:][::-1], 1):
    print(f"   {k}. {p['a1']}  ↔  {p['a2']}  (dist: {p['d']:.2f})")

print("\n👯 CARA GEMELA de cada alumno:")
for i, nombre in enumerate(nombres_alumnos):
    opciones = [(nombres_alumnos[j], matriz_dist[i][j]) for j in range(n) if j != i]
    gemelo, dist = min(opciones, key=lambda x: x[1])
    print(f"   {nombre:25s} → {gemelo:25s}  (dist: {dist:.2f})")